# 05. 가드레일(Guardrails)

> 에이전트도 가드레일 없이 운영하면 사고가 나요. PII 4가지 처리 전략과 `@before_agent`/`@after_agent` 검증, Defense-in-Depth 원칙을 코드로 적용해요.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. **PII 보호**의 4가지 전략(redact, mask, hash, block)을 이해하고 `PIIMiddleware`를 적용할 수 있어요
2. 커스텀 정규 표현식으로 한국 전화번호, API 키 등 비즈니스 맞춤형 PII 탐지기를 만들 수 있어요
3. `@before_agent` / `@after_agent` 데코레이터와 `AgentMiddleware` 클래스로 커스텀 가드레일을 구현할 수 있어요
4. **계층화된 방어(Defense in Depth)** 전략으로 여러 가드레일을 올바른 순서로 조합할 수 있어요

## 사전 지식

- `04-Prebuilt-Middleware.ipynb`에서 배운 `SummarizationMiddleware`, `PIIMiddleware` 등 내장 미들웨어 개념
- LangChain V1 미들웨어 기본 구조 (`before_model`, `after_model` 훅)

## 가드레일이란?

가드레일(Guardrails)은 에이전트 실행 중 주요 지점에서 콘텐츠를 **검증하고 필터링**하여 안전하고 규정을 준수하는 AI 애플리케이션을 구축할 수 있도록 도와줘요.

> 🔑 **핵심 개념**: 가드레일을 **고속도로 방호벽**에 비유할 수 있어요. 운전자(에이전트)가 정상 주행할 때는 존재를 못 느끼지만, 차선을 이탈(부적절한 입/출력)할 때 즉시 방향을 교정해줘요. 가드레일이 없으면 에이전트가 개인정보를 유출하거나, 유해한 답변을 생성하거나, 금지된 작업을 수행할 수 있어요.

가드레일을 통해:
- 부적절한 입력을 사전에 **차단**하고
- 민감한 개인정보(PII)를 **보호**하며
- 출력 **품질과 안전성**을 보장할 수 있어요

하네스 관점에서 보면 가드레일은 5가지 사고 도구 중 **Constrain**(못 하게 막기), **Verify**(결과 확인하기), **HITL**(위험 지점에서 사람에게 묻기)을 코드로 구현한 계층이에요.

### 두 가지 접근 방식

| 방식 | 원리 | 속도 / 비용 | 적합한 용도 |
|------|------|------------|------------|
| **결정론적(Deterministic)** | 정규표현식, 키워드 매칭 | 밀리초 / 저비용 | 금지어 필터, PII 패턴 탐지 |
| **모델 기반(Model-based)** | LLM 또는 분류기 활용 | 초 단위 / 고비용 | 감정 분석, 안전성 평가, 품질 검증 |

> 💡 **실무 팁**: 결정론적 가드레일은 빠르고 저렴하지만 미묘한 위반을 놓칠 수 있어요. 모델 기반 가드레일은 정교하지만 비용이 높아요. 실무에서는 두 가지를 **계층화하여 조합**하는 것이 모범 사례예요 -- 마치 공항에서 금속탐지기(결정론적)와 보안 요원(모델 기반)을 함께 배치하는 것처럼요.

### 전체 아키텍처 다이어그램

```mermaid
flowchart TD
    A["사용자 입력<br>User Input"] --> B["계층 1: 결정론적 필터<br>금지어 / 속도 제한"]
    B -->|"차단"| Z1["거부 응답<br>즉시 반환"]
    B -->|"통과"| C["계층 2: PII 보호<br>redact / mask / hash / block"]
    C -->|"차단"| Z2["차단 응답<br>즉시 반환"]
    C -->|"정제된 입력"| D["에이전트 실행<br>create_agent"]
    D --> E["계층 3: HITL<br>민감한 도구 승인"]
    E -->|"거부"| Z3["취소 응답<br>즉시 반환"]
    E -->|"승인"| F["도구 실행<br>ToolNode"]
    F --> G["최종 응답 생성"]
    G --> H["계층 4: 모델 기반 검증<br>SAFE / UNSAFE"]
    H -->|"UNSAFE"| Z4["안전 경고<br>응답 대체"]
    H -->|"SAFE"| I["사용자에게 전달"]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e

    class A input
    class B,C,H process
    class D,E,F,G process
    class Z1,Z2,Z3,Z4 error
    class I output
```

## 환경 설정

In [ ]:
# 환경 변수를 .env 파일에서 불러와요
from dotenv import load_dotenv

load_dotenv()

In [ ]:
# LangSmith 추적 설정 (선택사항 - 가드레일 트리거 상황을 모니터링할 수 있어요)
import os

# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-V1-Guardrails"

## 1. PII 보호 - 4가지 전략

개인 식별 정보(PII, Personally Identifiable Information)는 이메일, 신용카드 번호, IP 주소 등 사용자를 특정할 수 있는 정보예요. LangChain의 `PIIMiddleware`는 이러한 정보를 탐지하여 다음 4가지 전략으로 처리해요:

> 🔁 **복습 연결**: `PIIMiddleware`의 유형·전략 옵션 카탈로그는 `04-Prebuilt-Middleware.ipynb` §2에서 다뤘어요. 여기서는 같은 미들웨어를 **가드레일(입력·출력 방어선)의 첫 계층**으로 배치하는 관점에 집중합니다.

| 전략 | 설명 | 변환 예시 | 적합한 상황 |
|------|------|----------|------------|
| `redact` | 완전히 제거 | `john@test.com` -> `[REDACTED_EMAIL]` | 최대 보안이 필요할 때 |
| `mask` | 일부만 표시 | `4532-1234-5678-9010` -> `****-****-****-9010` | 부분 확인이 필요할 때 |
| `hash` | 결정론적 해시로 대체 (추적 가능) | `192.168.1.1` -> `<ip_hash:c5eb5a4c>` | 보안 감사 로그가 필요할 때 |
| `block` | 탐지 시 예외 발생 (처리 중단) | -- 처리 자체를 차단 -- | 절대 모델에 전달되면 안 될 때 |

> 💡 **핵심 정리**: `hash` 전략은 값은 숨기면서도 동일한 값이면 동일한 해시가 나와서 **로그 추적**이 가능해요. 보안 감사가 필요한 경우에 유용하게 쓰여요.

> 💡 **실무 팁**: `apply_to_input=True`는 모델에 전달되는 입력을 정제하고, `apply_to_output=True`는 모델의 응답에서도 PII를 제거해요. 민감한 환경에서는 **둘 다** 활성화하는 것을 권장해요.

> ⚠️ **자주 하는 실수**: `strategy="block"`은 PII가 감지되면 **요청 자체를 차단**(예외 발생)해요. 챗봇처럼 사용자 경험이 중요한 곳에서는 `"redact"` 또는 `"mask"`를 사용하는 것이 더 자연스러워요.

In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 기본 에이전트 구성을 위한 임포트
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from langchain.agents.middleware import PIIMiddleware

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: PIIMiddleware 임포트
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 내장 PII 타입 정리

LangChain이 제공하는 검증된 내장 PII 패턴이에요:

| 타입 | 설명 | 예시 패턴 |
|------|------|----------|
| `email` | 이메일 주소 | `user@domain.com` |
| `credit_card` | 신용카드 번호 (Luhn 알고리즘 검증 포함) | `4532-1234-5678-9010` |
| `ip` | IP 주소 (IPv4, IPv6) | `192.168.1.1`, `::1` |
| `mac_address` | MAC 주소 | `00:1A:2B:3C:4D:5E` |
| `url` | URL | `https://example.com` |

## 2. 커스텀 PII 탐지기

내장 PII 타입으로 충분하지 않을 때는 **정규 표현식(regex)**으로 커스텀 패턴을 정의할 수 있어요. 한국 전화번호, 사원번호 등 비즈니스에 특화된 패턴을 추가하는 데 유용해요.

> ⚠️ **자주 하는 실수**: `PIIMiddleware`의 첫 번째 인자는 탐지기의 **이름**이에요. `detector=` 파라미터에 정규 표현식을 따로 전달해야 해요. 내장 타입(`email`, `credit_card` 등)은 `detector=`를 생략하면 내장 패턴을 사용해요.

> 🔑 **핵심 개념**: `strategy="block"`은 탐지 즉시 예외를 발생시켜요. 절대로 모델에 전달되면 안 되는 비밀 정보에 적합해요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 3. @before_agent 가드레일 - 입력 검증

`@before_agent` 데코레이터를 사용하면 에이전트 실행이 **시작되기 전**에 요청을 검증할 수 있어요. 금지어 필터링, 인증 확인, 속도 제한 등 처리 전에 부적절한 요청을 사전 차단하는 데 유용해요.

**핵심 패턴:**
- `can_jump_to=["end"]`: 이 훅이 `jump_to="end"`를 반환할 수 있도록 허용해요
- `return None`: 정상 진행 (아무것도 하지 않음)
- `return {"messages": [...], "jump_to": "end"}`: 에이전트를 즉시 종료하고 응답 반환

> 💡 **핵심 정리**: `jump_to="end"` 없이 메시지만 반환하면 에이전트가 계속 실행돼요. 실제로 차단하려면 반드시 `"jump_to": "end"`를 함께 반환해야 해요.

> 🔍 **`can_jump_to`는 왜 필요할까요?**: `create_agent`는 그래프를 컴파일할 때 각 훅이 어디로 분기할 수 있는지 미리 알아야 조건부 엣지를 그려둘 수 있어요. `can_jump_to=["end"]`는 "이 훅은 흐름을 `end`로 건너뛸 수 있다"고 그래프에 **선언**하는 거예요. 이 선언이 없으면 런타임이 `jump_to="end"`를 받아도 연결된 경로가 없어 무시해요. 즉 `can_jump_to`(선언)와 `jump_to`(실제 반환)는 **짝**으로 동작해요.

In [ ]:
from langchain.agents.middleware import before_agent, AgentState
from typing import Any

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: @before_agent 데코레이터 임포트
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from IPython.display import Image, display

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → agent → tools → agent → ... → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 정상 요청은 가드레일을 통과해서 실행돼요
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 4. AgentMiddleware 클래스 - 재사용 가능한 가드레일

`@before_agent` 데코레이터는 간단한 로직에 좋지만, **설정 가능한 파라미터**가 필요하거나 **여러 훅을 한 클래스에서 관리**해야 할 때는 `AgentMiddleware`를 상속받아 클래스로 구현해요.

| 방식 | 사용 시기 | 장점 |
|------|----------|------|
| 데코레이터 (`@before_agent`) | 단순한 검증 로직 | 간결한 코드 |
| 클래스 (`AgentMiddleware` 상속) | 설정 파라미터 필요, 상태 유지 | 재사용성, 여러 훅 조합 |

> 💡 **실무 팁**: 클래스 기반 미들웨어는 패키지로 배포하거나 여러 에이전트에서 재사용할 때 훨씬 관리하기 좋아요. 프로덕션 환경에서는 클래스 방식을 더 추천해요.

In [ ]:
from langchain.agents.middleware import AgentMiddleware, hook_config

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: AgentMiddleware와 hook_config 임포트
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 5. @after_agent 가드레일 - 출력 검증

`@after_agent` 데코레이터를 사용하면 사용자에게 응답이 전달되기 **전**에 최종 출력을 검증할 수 있어요. 모델 기반 안전성 검사, 품질 검증, 규정 준수 확인 등에 활용해요.

> ⚠️ **자주 하는 실수**: `@after_agent`는 에이전트 실행 **후**에 호출되므로, 이미 LLM 비용이 발생한 상태예요. 따라서 입력에서 처리할 수 있는 것은 `@before_agent`에서 먼저 처리하고, 복잡한 의미론적 검증만 `@after_agent`에서 수행하는 것이 비용 효율적이에요. 즉, **결정론적 필터를 앞에 두고, LLM 기반 검증은 뒤에 둔다**는 하네스 원칙이 여기에도 적용됩니다.

> 🔑 **핵심 개념**: `@after_agent`에서 메시지를 반환하면 기존 마지막 메시지 **뒤에 추가**돼요. `jump_to="end"`와 함께 사용하면 응답을 **대체**할 수 있어요.

In [ ]:
from langchain.agents.middleware import after_agent
from langchain_core.messages import AIMessage

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: after_agent 데코레이터와 AIMessage 임포트
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 6. Human-in-the-Loop 가드레일

민감한 작업 실행 전에 **사람의 승인**을 요구하는 것은 가장 강력한 가드레일 중 하나예요. `HumanInTheLoopMiddleware`를 사용하면 특정 도구 호출 전에 실행을 일시 중지하고 사람의 결정을 기다릴 수 있어요.

> 🔁 **복습 연결**: `HumanInTheLoopMiddleware` 자체와 approve/edit/reject/respond 결정 타입은 `02-Human-In-The-Loop-V1.ipynb`에서 자세히 배웠어요. 여기서는 그 HITL을 **여러 방어선 중 하나(사람 승인 계층)**로 재배치하는 관점만 봅니다.

**HITL이 유용한 경우:**
- 금융 거래 및 환불 처리
- 이메일 / 메시지 전송
- 데이터 삭제 또는 수정
- 비즈니스에 중요한 외부 API 호출

> 💡 **실무 팁**: `HumanInTheLoopMiddleware`는 내부적으로 LangGraph의 `interrupt`를 사용해요. 따라서 반드시 `checkpointer=InMemorySaver()`를 설정해야 하고, 재개(resume)할 때 동일한 `thread_id`를 사용해야 해요.

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: HITL 관련 임포트
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → agent → tools → agent → ... → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: --- 2단계: 승인 후 재개 ---
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 7. 계층화된 방어 (Defense in Depth)

실무 환경에서는 단일 가드레일로 충분하지 않아요. 여러 가드레일을 **올바른 순서로** 조합하면 한 계층이 실패해도 다른 계층이 보호해요.

### 권장 가드레일 순서

```
빠름 ←─────────────────────────────────────→ 느림
저비용 ←────────────────────────────────── 고비용

1. 결정론적 필터  →  2. PII 보호  →  3. HITL  →  4. 모델 기반 검증
   (regex, ms)       (regex, ms)    (사람 개입)    (LLM, 초 단위)
```

> 💡 **핵심 정리**: 빠른 검사를 먼저 배치하는 이유가 있어요. 금지어로 차단되어야 할 요청이 LLM 안전성 검사까지 도달하면 비용이 낭비되고, 예상치 못한 응답이 생성될 위험도 있어요. **입구를 좁혀서 비용을 줄이는** 것이 핵심이에요.

> 단, HITL은 항상 "모델 기반 검증보다 앞"이라는 뜻은 아니에요. HITL은 보통 결제·삭제·전송처럼 **외부 상태가 바뀌기 직전**에 둡니다. 순서는 워크플로 위험도에 맞게 조정하되, 싼 결정론적 검사는 가능한 앞에 두세요.

> 💡 **실무 팁**: 미들웨어 배열은 `before_*` 훅은 배열 **정순**으로, `after_*` 훅은 배열 **역순**으로 실행돼요. `[A, B, C]` 순서로 등록하면: before는 `A→B→C`, after는 `C→B→A` 순서예요.

In [ ]:
import uuid

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → agent → tools → agent → ... → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: === 테스트 2: 정상 검색 요청 → 모든 계층 통과 ===
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 8. 속도 제한(Rate Limiting) 가드레일

짧은 시간 내에 너무 많은 요청이 들어오면 서비스가 불안정해질 수 있어요. `AgentMiddleware`를 사용해서 요청 수를 제한하는 가드레일을 만들 수 있어요.

> ⚠️ **자주 하는 실수**: 아래 예제는 **인스턴스 내 메모리**에서만 카운터를 유지해요. 여러 서버 인스턴스가 있는 프로덕션 환경에서는 **Redis 등 외부 저장소**를 사용해야 해요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 9. TODO 실습: 맞춤형 가드레일 만들기

아래 코드를 완성해서 한국어 금융 서비스용 가드레일을 만들어 보세요!

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 금융 서비스용 맞춤형 가드레일 완성하기
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **PIIMiddleware**: 이메일, 신용카드, IP 등 개인정보를 `redact` / `mask` / `hash` / `block` 4가지 전략으로 처리해요
- **커스텀 PII 탐지기**: `detector=r"패턴"` 으로 한국 전화번호, API 키 등 맞춤형 PII 패턴을 정의할 수 있어요
- **@before_agent**: 에이전트 실행 전 입력을 검증하고, `jump_to="end"`로 즉시 종료할 수 있어요
- **@after_agent**: 에이전트 실행 후 출력을 검증하고, LLM 기반 SAFE/UNSAFE 판정을 적용할 수 있어요
- **AgentMiddleware 클래스**: 설정 파라미터와 상태를 유지하는 재사용 가능한 미들웨어를 만들 수 있어요
- **Defense in Depth**: 빠른 결정론적 필터 → PII 보호 → HITL → 모델 기반 검증 순서로 계층화해요
- **하네스 관점**: 가드레일은 Constrain / Verify / HITL을 실행 루프에 끼워 넣는 방법이에요
- **성능 최적화**: 밀리초 단위 regex 검사를 앞에, 초 단위 LLM 검사를 뒤에 배치하여 비용을 절감해요

## 다음 노트북 예고

다음 `07_Memory/01-Short-Term-Memory.ipynb`에서는 **단기 메모리(Checkpointer)**를 먼저 배워요. `thread_id`로 대화 세션을 구분하고, 메시지 누적·삭제·요약으로 컨텍스트를 관리하는 흐름을 정리한 뒤, 그 다음 `02-Long-Term-Memory.ipynb`에서 사용자 간 공유 가능한 Store API로 확장해요. 지금까지 배운 가드레일로 보호된 에이전트에게 단계적으로 **"대화를 기억하는" 능력**을 더해주는 흐름이에요.